In [1]:
import fitz
import re, os
from pathlib import Path
import random
import time
from collections import Counter
from io import StringIO
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

NUM_LINE_RE = re.compile(
    r"^\s*[\(\[\{]?\s*[₹$€£]?\s*[-+]?"
    r"\d[\d,]*(?:\.\d+)?%?"
    r"\s*[\)\]\}]?\s*$"
)
SECTION_RE = re.compile(
    r"\b(?:financial\s+statements|standalone|consolidated|balance\s+sheet|profit\s+(?:&|and)\s+loss|cash\s+flow)\b",
    re.I
)
HEADLINE_RE = re.compile(r"\b(?:Standalone(?:\s+\S+){0,4}|Consolidated(?:\s+\S+){0,4})?\s+(?:Balance\s+Sheet|Profit\s+(?:and|&)\s+Loss|Cash\s+Flows?|Changes\s+In\s+Equity)\b", re.I)


In [5]:
def add_sequential_features(df, pdf_col="pdf", page_col="page_n", side_col="side"):
    df = df.copy()

    sort_cols = [pdf_col, page_col]
    if side_col in df.columns:
        sort_cols.append(side_col)
    df = df.sort_values(sort_cols).reset_index(drop=True)


    for c in [
        "rolling_median", "rolling_std", "robust_zscore",
        "run_length", "peak_prominence", "valley_prominence",
        "adjacent_page_similarity"
    ]:
        df[c] = np.nan

    df["high_activity"] = False
    df["peak_flag"] = False
    df["valley_flag"] = False

    for _, g in df.groupby(pdf_col):
        idx = g.index
        s = g["structural_score"]

        median = s.rolling(10, center=True, min_periods=1).median()
        std = s.rolling(10, center=True, min_periods=2).std().fillna(0)

        mad = s.rolling(10, center=True, min_periods=3).apply(
            lambda x: np.median(np.abs(x - np.median(x))), raw=True
        ).replace(0, np.nan)

        z = (
            0.6745 * (s - median) / mad
        ).replace([np.inf, -np.inf], np.nan).fillna(0)

        high = s >= median

        run = []
        current = 0
        for x in high:
            current = current + 1 if x else 0
            run.append(current)

        smooth = s.rolling(7, center=True, min_periods=1).median().to_numpy()

        peaks, pp = find_peaks(smooth, prominence=0.03, distance=2)
        valleys, vp = find_peaks(-smooth, prominence=0.03, distance=2)

        peak_prom = np.zeros(len(g))
        valley_prom = np.zeros(len(g))
        peak_flag = np.zeros(len(g), dtype=bool)
        valley_flag = np.zeros(len(g), dtype=bool)

        peak_prom[peaks] = pp["prominences"]
        valley_prom[valleys] = vp["prominences"]
        peak_flag[peaks] = True
        valley_flag[valleys] = True

        similarity = (1 - s.diff().abs()).clip(0, 1)

        df.loc[idx, "rolling_median"] = median.to_numpy()
        df.loc[idx, "rolling_std"] = np.round(std.to_numpy(),2)
        df.loc[idx, "robust_zscore"] = np.round(z.to_numpy(),2)
        df.loc[idx, "high_activity"] = high.to_numpy()
        df.loc[idx, "run_length"] = run
        df.loc[idx, "peak_prominence"] = peak_prom
        df.loc[idx, "peak_flag"] = peak_flag
        df.loc[idx, "valley_prominence"] = valley_prom
        df.loc[idx, "valley_flag"] = valley_flag
        df.loc[idx, "adjacent_page_similarity"] = similarity.to_numpy()

    return df

def gen_prb(st,et, n_lines=25):
    rng = random.Random(42) #seed
    lines = sorted(
        rng.uniform(st,et)
        for _ in range(n_lines)
    )
    return lines

def page_layout_metadata(page, ref_width, ref_height):
    w = page.rect.width
    h = page.rect.height

    spread = (w / ref_width) > 1.6 and (h / ref_height) > 0.7

    return {
        "width": int(w),
        "height": int(h),
        # "area": round(w * h, 2),
        # "rotation": page.rotation,
        "spread": spread,
        "split_x": round(w * 0.5, 2) if spread else None,
    }

def page_content_metadata_v1(page, layout, region):

    # margin_x = (region["x1"] - region["x0"]) * 0.05
    margin_y = (region["y1"] - region["y0"]) * 0.15

    left = region["x0"]
    right = region["x1"]
    top = region["y0"]
    bottom = region["y1"] - margin_y
    
    headline_bottom = (region["y0"] + (region["y1"] - region["y0"])*0.35 )

    numeric_data = []
    headline_text = []
    dirs = Counter()

    # char_count = 0
    line_count = 0
    block_count = 0
    blocks = page.get_text("dict")["blocks"]

    for block in blocks:
        if block["type"] != 0:
            continue
        
        block_count += 1
        
        for line in block["lines"]:

            x0, y0, x1, y1 = line["bbox"]

            # region restriction
            if  x1 < left or y1 < top or x0 > right or y0 > bottom:
                continue
            
            line_count += 1
            spans = line["spans"]
            dirs[tuple(map(round, line["dir"]))] += 1
            text = "".join(span["text"] for span in spans).strip()

            if not text:
                continue
            # char_count += len(text)
            
            # filter the 35% dude
            if y0 <= headline_bottom:
                headline_text.append(text)
            
            
            if NUM_LINE_RE.match(text):
                numeric_data.append(
                    {
                        "text": text,
                        "bbox": (x0, y0, x1, y1),
                        #imp
                        "cx": (x0 + x1) / 2,
                    }
       
                )

            # matches = SECTION_RE.findall(text)
            # if matches:
            #     segment_lines.extend(matches)

    if dirs:

        dominant = dirs.most_common(1)[0][0]

        text_dir = {
            (1, 0): "n",
            (0, -1): "90_cc",
            (0, 1): "90_c",
            (-1, 0): "ud",
        }.get(dominant, "n")

    else:
        text_dir = None

    numeric_count = len(numeric_data)
    headline_text = re.sub(r"\s+", " ", re.sub(r"[^A-Za-z0-9&\s]+", "", " ".join(headline_text))).strip()
    headline_matches = HEADLINE_RE.findall(headline_text)
    
    return {
        **layout,
        # "char_count": char_count,
        "line_count": line_count,
        "block_count": block_count,
        "t_dir": text_dir,
        "numeric_lines": numeric_data,
        "numeric_count": numeric_count,
        # "segment_lines": segment_lines,
        # "segment_count": len(segment_lines),
        "headline_data":headline_text,
        "head_match": headline_matches,
        "head_bool": bool(headline_matches),
        "probe_x": None if not numeric_count else gen_prb(region["x0"], region["x1"]),
        "probe_y": None if not numeric_count else gen_prb(region["y0"], region["y1"])
    }

def page_probe_metadata_v1(metadata):

    probes = metadata["probe_x"]
    yprobes = metadata["probe_y"]
    
    # print(f"X_PROBE: {probes} || Y_PROBE: {yprobes}")
    
    if not probes:
        metadata.update({
            "probe_x_hits":[],
            "probe_y_hits":[],
            # "max_hits":0,
            # "total_hits":0,
            # "hits_1":0,
            # "const_hit":0
        })
        
        return metadata
    
    xhits = [0] * len(probes)
    yhits = [0] * len(probes)
    # For X
    for item in metadata["numeric_lines"]:
        x0, _, x1, _ = item["bbox"]
        for i, px in enumerate(probes):
            if x0 <= px <= x1:
                xhits[i] += 1
    # For Y       
    for item in metadata["numeric_lines"]:
        _, y0, _, y1 = item["bbox"]
        for i, px in enumerate(yprobes):
            if y0 <= px <= y1:
                yhits[i] += 1

    metadata["probe_x_hits"] = xhits
    metadata["probe_y_hits"] = yhits
    # metadata["max_hits"] = max(hits)
    # metadata["total_hits"] = sum(hits)
    # metadata["hits_1"] = sum(h > 1 for h in hits)
    # metadata["cons_hit"] = consecutive_hits(hits, threshold=1)
    
    return metadata

def consecutive_hits(hits, threshold=0):
    longest = 0
    current = 0
    for h in hits:
        if h > threshold:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest
    

In [3]:
path = r"PDF_TO_RUN.csv"
df1 = pd.read_csv(path)
df1.head(5)

files = df1.pdf_name.to_list()

In [6]:
folder_path = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"

probe_data = []
pdf_data = []

# files = os.listdir(folder_path)
total_files = len(files)



for idx, file in enumerate(files):
    print(f"{idx}/{total_files}: {file}")
    pdf_path = os.path.join(folder_path, file)
    pdf_file = Path(pdf_path)

    start_t = time.perf_counter()

    with fitz.open(pdf_path) as doc:

        total_pages = doc.page_count

        # Reference dimensions for spread detection
        ref_page = doc[0]
        rw = ref_page.rect.width
        rh = ref_page.rect.height

        for page_no, page in enumerate(doc):

            layout = page_layout_metadata(page, rw, rh)

            w = layout["width"]
            h = layout["height"]

            page_number = page_no + 1

            if layout["spread"]:

                split_x = layout["split_x"]

                regions = (
                    (
                        {"x0": 0, "y0": 0, "x1": split_x, "y1": h},
                        "L",
                    ),
                    (
                        {"x0": split_x, "y0": 0, "x1": w, "y1": h},
                        "R",
                    ),
                )

            else:

                regions = (
                    (
                        {"x0": 0, "y0": 0, "x1": w, "y1": h},
                        "S",
                    ),
                )

            for region, side in regions:

                metadata = {
                    "pdf": pdf_file.stem,
                    "page_n": page_number,
                    "side": side,
                }

                metadata.update(
                    page_content_metadata_v1(
                        page,
                        layout,
                        region,
                    )
                )

                metadata = page_probe_metadata_v1(metadata)

                probe_data.append(metadata)

    elapsed = time.perf_counter() - start_t

    pdf_data.append(
        {
            "pdf_name": pdf_file.stem,
            "total_pages": total_pages,
            "file_size": os.path.getsize(pdf_path),
            "time_elapsed": elapsed,
        }
    )

0/1372: Ashnoor Textile Mills Ltd..pdf
1/1372: Ashutosh Paper Mills Ltd..pdf
2/1372: Asian Petroproducts & Exports Ltd..pdf
3/1372: Atharv Enterprises Ltd..pdf
4/1372: AD Manum Finance Ltd..pdf
5/1372: Atlas Cycles (Haryana) Ltd..pdf
6/1372: Axtel Industries Ltd..pdf
7/1372: Capricorn Systems Global Solutions Ltd..pdf
8/1372: Caprolactam Chemicals Ltd..pdf
9/1372: Ceeta Industries Ltd..pdf
10/1372: Chaman Lal Setia Exports Ltd..pdf
11/1372: Chennai Ferrous Industries Ltd..pdf
12/1372: Cityman Ltd..pdf
13/1372: Cochin Malabar Estates & Industries Ltd..pdf
14/1372: Country Condos Ltd..pdf
15/1372: Credo Brands Marketing Ltd..pdf
16/1372: Cubex Tubings Ltd..pdf
17/1372: Gillette India Ltd..pdf
18/1372: Gini Silk Mills Ltd..pdf
19/1372: Go Fashion (India) Ltd..pdf
20/1372: Gold Coin Health Foods Ltd..pdf
21/1372: Grandma Trading & Agencies Ltd..pdf
22/1372: Gujarat Lease Financing Ltd..pdf
23/1372: Gujarat Poly Electronics Ltd..pdf
24/1372: H S India Ltd..pdf
25/1372: HB Leasing & Finance 

In [7]:
df = pd.DataFrame(probe_data)
#X
df["max_x_hits"] = df["probe_x_hits"].apply(lambda x: max(x) if x else 0)
df["total_x_hits"] = df["probe_x_hits"].apply(lambda x: sum(x) if x else 0)
df["x_hits>1"] = df["probe_x_hits"].apply(lambda x : sum(i> 1 for i in x) if x else 0)
df["x_cons_hit"] = df["probe_x_hits"].apply(lambda x: consecutive_hits(x, threshold=1) if x else 0)
#Y
# df["max_y_hits"] = df["probe_y_hits"].apply(lambda x: max(x) if x else 0)
# df["total_y_hits"] = df["probe_y_hits"].apply(lambda x: sum(x) if x else 0)
# df["y_hits>1"] = df["probe_y_hits"].apply(lambda x : sum(i> 1 for i in x) if x else 0)
# df["y_cons_hit"] = df["probe_y_hits"].apply(lambda x: consecutive_hits(x, threshold=1) if x else 0)

df["numeric_score"] = df.groupby("pdf")["numeric_count"].rank(pct=True).round(1)
df["hits_score"] = df.groupby("pdf")["total_x_hits"].rank(pct=True).round(1)
df["cons_score"] = df.groupby("pdf")["x_cons_hit"].rank(pct=True).round(1)
df["line_score"] = df.groupby("pdf")["line_count"].rank(pct=True).round(1)
df["structural_score"] = (
    0.60 * df["numeric_score"]
    + 0.30 * df["hits_score"]
    + 0.10 * df["cons_score"]
    # + 0.20 * df["line_score"]
).round(1)

df = add_sequential_features(df)
ndf = df.drop(["spread","split_x","probe_x","probe_y","probe_x_hits","probe_y_hits","numeric_lines","headline_data"],axis = 1)

fp = Path(folder_path)
excel_path = f"{fp.stem}_HITS.xlsx"
df1 = pd.DataFrame(pdf_data)
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    ndf.to_excel(writer, sheet_name ="page_wise" ,index=False)
    df1.to_excel(writer,sheet_name ="pdf_wise",index=False)